# K5 trained on uncut Nexar videos (`whole`), s2024, read beside K5 (window)

Rule: `core/docs/final/KIM_RUN.md` §16, written before any number. **Printed, decides nothing:** K5 on `Nexar_window` stays the main model. One seed: **PRELIMINARY (n = 1)**. `nexar_test` is never touched. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first.

The row = K5's config with three changes (§16 WH1): corpus `Nexar_whole` (900 videos, 451 positive, video labels), `loss.mil_topk_pct` 5 → **16**, and `train.num_epochs` 116 (116 × 15 = 1,740 steps). Everything else is K5, including `model.temporal_window` 25 and DVS off (so no KNN cache is passed).

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | setup, preflight (corpus, bakes, K5, its reads) | no | 2-5 min |
| 4 | pytest on the report tool — **raises if red** | no | 1 min |
| 5 | train K5 (whole), s2024, 1,740 steps -> `outputs/kim_whole/K5/s2024` | yes | about as long as K5 |
| 6 | K5 (whole) on DoTA-CAP-dev, DADA whole, TAD, DAD (Nexar statistics, as K5) | yes | 20-30 min |
| 7 | `nexar_val`: K5 (whole); K5 (window) only if its §13 read is missing | yes | 10-20 min |
| 8 | `core.tools.k5_whole_report` -> **paste `REPORTS/k5_whole_report.md` to Claude** | no | 1 min |

A re-run skips what is on Drive. Thesis and Thesis-V2 are only read; everything new goes to Thesis-Final. If the runtime dies in step 5 the local checkpoint is lost and the run starts again with the same seed.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_whole, A2 bakes, Nexar caches, census, v2 pilot
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, K5 and its reads, DAD; every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert (REPO / 'core/tools/k5_whole_report.py').is_file(), 'this upload has no k5_whole_report -- re-upload kat-vad/'
assert '## 16. K5 trained on uncut Nexar videos' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §16 -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K5, WHOLE = constants.KIM_MAIN_ARM, constants.KIM_WHOLE_ROW
SEED     = constants.KIM_SEED                      # 2024
ENC      = constants.V5_ENCODER                    # V2-S
DATASET  = constants.NEXAR_DATASET
C_WHOLE, C_WINDOW = constants.NEXAR_CONSTRUCTION_WHOLE, constants.NEXAR_CONSTRUCTION_WINDOW
TOPK     = constants.KIM_WHOLE_MIL_TOPK_PCT        # 16 (§16 WH1)
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
NEXAR_SPLIT = constants.V2_SPLIT_NEXAR_VAL         # nexar_test is never read here
TAD_NAME = constants.TAD_DATASET
DAD_NAME, DAD_STRIDE = constants.DAD_DATASET, constants.DAD_STRIDE

V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
BUILD    = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n4' / 'build_report.json'
# Training (§16 WH1-WH4)
DATA     = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C_WHOLE}'
DEFS     = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C_WINDOW}'   # the definitions every Nexar row read with
# No KNN cache: DVS is off (theta = 1.0), so no filler is ever drawn (core/data/dataset.py:177)
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'                # nexar_train statistics, construction-free (WH3)
# K5 (window): the row it is compared with
KIM      = constants.OUTPUT_ROOT / constants.KIM_OUTPUT_DIRNAME
K5_RUN   = KIM / K5 / f's{SEED}'
K5_NEXAR = constants.OUTPUT_ROOT / constants.KIM_T2_OUTPUT_DIRNAME / 'eval' / 'nexar' / K5   # §13's nexar_val read
# Zero-shot inputs: the same bakes K5 was read on (kim_runs.ipynb, k5_dad.ipynb)
A2_DADA  = V2_CACHE / 'v2' / 'A2_S' / f'{T2NAME}_test_sources_nexar'
A2_DOTA  = V2_CACHE / 'v2' / 'A2_S' / f'{CAP_DEV}_s3_from_s1_nexar'
A2_TAD   = V2_CACHE / 'v2' / 'A2_S' / f'{TAD_NAME}_s8_nexar'
A2_DAD   = constants.CACHE_ROOT / 'v2' / 'A2_S' / f'{DAD_NAME}_s{DAD_STRIDE}_nexar'
T2       = constants.DATA_ROOT / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
TAD      = constants.DATA_ROOT / TAD_NAME
DAD_DATA = Path(DRIVE_F) / 'data' / f'{DAD_NAME}_s{DAD_STRIDE}'
# Nexar val
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
NEXAR_X  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET
NEXAR_U  = V2_CACHE / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'
PILOT    = V2_OUT / 'v2' / 'REPORTS' / 'nexar_val_ignore' / 'pilot_whole_A3'   # printed only, other host
# Outputs
ROOT     = constants.OUTPUT_ROOT / constants.KIM_WHOLE_OUTPUT_DIRNAME
RUN      = ROOT / K5 / f's{SEED}'
CKPT     = RUN / 'checkpoint_last.pt'
EVAL     = ROOT / 'eval'
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/k5_whole')
LOCAL.mkdir(parents=True, exist_ok=True)

PLAN = json.loads(BUILD.read_text())['plan'][C_WHOLE]
STEPS, EPOCHS = PLAN['steps'], PLAN['epochs']      # 1,740 = 116 x 15
assert STEPS == constants.NEXAR_STEP_BUDGET, PLAN

INPUTS = (('build report', BUILD), ('Nexar_whole', DATA / constants.LABELS_TRAIN_FILENAME), ('Nexar_window defs', DEFS / constants.DEFS_FILENAME),
          ('Nexar A2 bake', A2_NEXAR / constants.V2_INPUT_MANIFEST_FILENAME),
          ('K5 checkpoint', K5_RUN / 'checkpoint_last.pt'), ('K5 verified', K5_RUN / 'verified.json'),
          ('K5 config', K5_RUN / 'config.yaml'),
          ('DADA A2 bake', A2_DADA), ('DoTA-CAP A2 bake', A2_DOTA), ('TAD A2 bake', A2_TAD), ('DAD A2 bake', A2_DAD),
          ('T2 meta', T2 / constants.META_FILENAME), ('T2 census', T2_CENSUS), ('DADA annotation', ANNOT),
          ('DoTA s1', DOTA_S1), ('TAD labels', TAD / constants.FRAME_LABELS_TEST_FILENAME),
          ('DAD labels', DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME),
          ('Nexar census', CENSUS), ('Nexar CLIP s1', NEXAR_X), ('Nexar V2-S s1', NEXAR_U),
          ('K5 DoTA read', KIM / 'eval' / 'dota' / K5), ('K5 DADA read', KIM / 'eval' / 'dada' / K5),
          ('K5 TAD read', KIM / 'eval' / 'tad' / K5 / f's{SEED}' / 'results.json'),
          ('K5 DAD read', KIM / 'eval' / constants.DAD_OUTPUT_SET / K5 / f's{SEED}' / 'results.json'))
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(INPUTS)} inputs found | steps {STEPS} ({EPOCHS} epochs) | output {ROOT}')
print(f'K5 nexar_val read from §13: {"found" if K5_NEXAR.exists() else "MISSING -- step 7 reads it"} | '
      f'v2 pilot: {"found" if PILOT.exists() else "missing (not printed)"}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 5-7 need one'
df -h /content | tail -1

In [ ]:
import torch
import yaml

from core.config import Config                       # noqa: E402
from core.data.dataset_files import TRAIN_IDS_FILENAME  # noqa: E402
from core.data.v2_splits import load_split           # noqa: E402
from core.tools.dada_whole_eval import test_sources  # noqa: E402
from core.tools.v5_stage1 import _flat               # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (only ``ids`` + json/npz manifests if given) or file to VM disk once (bit-identical; C2).'''
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    if ids is None:
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    else:
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def ckpt_step(path):
    '''The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).'''
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def is_verified(drive_dir, steps):
    rec, ckpt = drive_dir / 'verified.json', drive_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == steps and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def manifest(d):
    return json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())


# Preflight on small files only (C39)
k5_cfg = _flat(yaml.safe_load((K5_RUN / 'config.yaml').read_text()))
assert k5_cfg['kim.enabled'] and k5_cfg['kim.prototypes'] and k5_cfg['kim.edge_mask'], 'K5_RUN is not K5'
assert k5_cfg['dvs.theta'] == 1.0 and k5_cfg['data.dataset'] == DATASET, 'K5_RUN is not K5 on Nexar'
assert k5_cfg['loss.mil_topk_pct'] == constants.DADA_ORIGIN_MIL_TOPK_PCT and k5_cfg['model.temporal_window'] == 25, k5_cfg
k5_steps = json.loads((K5_RUN / 'verified.json').read_text())['step']
assert is_verified(K5_RUN, k5_steps) and metrics_last(K5_RUN / 'metrics.jsonl')['global_step'] == k5_steps, 'K5: J10'
built = json.loads(BUILD.read_text())['constructions'][C_WHOLE]
assert (built['train_items'], built['train_abnormal']) == (900, 451), built
train_ids = (DATA / TRAIN_IDS_FILENAME).read_text().split()
assert sorted(train_ids) == sorted(load_split(constants.V2_SPLIT_NEXAR_TRAIN)), 'Nexar_whole is not nexar_train'
labels = json.loads((DATA / constants.LABELS_TRAIN_FILENAME).read_text())
assert (len(labels), sum(labels.values())) == (900, 451), (len(labels), sum(labels.values()))
assert (DATA / constants.DEFS_FILENAME).read_text() == (DEFS / constants.DEFS_FILENAME).read_text(), 'defs differ: whole vs window'
SHA = manifest(A2_NEXAR)['train_ids_sha1']
for name, d in (('Nexar', A2_NEXAR), ('DADA', A2_DADA), ('DoTA-CAP', A2_DOTA), ('TAD', A2_TAD), ('DAD', A2_DAD)):
    m = manifest(d)
    assert (m['crn'], m['motion'], m['train_ids_sha1']) == (constants.V2_OFF, ENC, SHA), f'{name} bake: {m}'
DADA_IDS = test_sources(T2)
CAP_IDS = sorted(load_split(CAP_DEV))
NEXAR_IDS = sorted(load_split(NEXAR_SPLIT))
TRAIN_IDS = sorted(load_split(constants.V2_SPLIT_NEXAR_TRAIN))
TAD_IDS = sorted(json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
DAD_IDS = sorted(json.loads((DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
assert len(DADA_IDS) == 370 and len(TAD_IDS) == constants.V2_TADZ_VIDEOS and len(DAD_IDS) == constants.DAD_TEST_VIDEOS
for name, d, ids in (('Nexar train bake', A2_NEXAR, TRAIN_IDS), ('DADA bake', A2_DADA, DADA_IDS),
                     ('DoTA-CAP bake', A2_DOTA, CAP_IDS), ('TAD bake', A2_TAD, TAD_IDS), ('DAD bake', A2_DAD, DAD_IDS),
                     ('Nexar CLIP', NEXAR_X, NEXAR_IDS), ('Nexar V2-S', NEXAR_U, NEXAR_IDS)):
    miss = [v for v in ids if not (d / f'{v}.npy').exists()]
    assert not miss, f'{len(miss)} of {name} missing: {miss[:5]}'
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in DEFS.glob('*.json'):
    shutil.copy2(f, L_DEFS / f.name)
print(f'preflight OK: K5 step {k5_steps} | train {len(TRAIN_IDS)} | DoTA-CAP-dev {len(CAP_IDS)}, DADA {len(DADA_IDS)}, '
      f'TAD {len(TAD_IDS)}, DAD {len(DAD_IDS)}, {NEXAR_SPLIT} {len(NEXAR_IDS)} | one set of statistics')

## 4. Tests on the report tool (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_k5_whole_report.py'])

## 5. Train K5 (whole), s2024 (§16 WH1-WH2)

**Watch the log: past 2 s/step, stop and paste it.** Then J10 and the config diff against K5: only `loss.mil_topk_pct` and `train.num_epochs` may differ (the corpus is a CLI path, checked in the preflight).

In [ ]:
K5_SETS = ['kim.enabled=true', 'kim.prototypes=true', 'kim.edge_mask=true', 'dvs.theta=1.0', 'dvs.theta_ego=1.0']
SETS = ['train.stage=2', f'train.seed={SEED}', f'train.num_epochs={EPOCHS}', 'train.amp=true',
        f'data.dataset={DATASET}', f'loss.mil_topk_pct={TOPK}',
        'kip.enabled=false', 'loss.dvs_anchor_mode=span', 'data.is_egocentric=false',
        f'v2.motion={ENC}', f'model.motion_dim={constants.V5_GRID_DIM}',
        f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}', *K5_SETS]
DEFAULTS = _flat(Config().to_dict())
ALLOWED = {'loss.mil_topk_pct', 'train.num_epochs'}


def config_diff(base, got, allowed):
    '''Keys that differ beyond ``allowed``; a key added after ``base`` trained may sit at its default.'''
    out = {}
    for k in set(base) | set(got):
        if base.get(k) == got.get(k) or k in allowed:
            continue
        if k not in base and got.get(k) == DEFAULTS.get(k):
            continue
        if k not in got and base.get(k) == DEFAULTS.get(k):
            continue
        out[k] = (base.get(k), got.get(k))
    return out


def check_run():
    payload = torch.load(CKPT, map_location='cpu', weights_only=False)
    assert payload['global_step'] == metrics_last(RUN / 'metrics.jsonl')['global_step'] == STEPS, 'J10'
    assert any(k.startswith('kim.') for k in payload['model']), 'no KIM weights in the checkpoint'
    got = _flat(yaml.safe_load((RUN / 'config.yaml').read_text()))
    diff = config_diff(k5_cfg, got, ALLOWED)
    assert not diff, f'K5 (whole) differs from K5 beyond {sorted(ALLOWED)}: {diff}'
    assert got['loss.mil_topk_pct'] == TOPK and got['train.num_epochs'] == EPOCHS and got['model.temporal_window'] == 25, got
    print(f'K5 (whole): J10 {STEPS} | == K5 except mil_topk_pct {k5_cfg["loss.mil_topk_pct"]} -> {TOPK}, '
          f'num_epochs {k5_cfg["train.num_epochs"]} -> {EPOCHS}')


if is_verified(RUN, STEPS):
    print(f'K5 (whole) s{SEED}: verified on Drive -- not retrained')
else:
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local = LOCAL / 'run'
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    t0 = time.time()
    run([sys.executable, '-m', 'core.train', *[x for s in SETS for x in ('--set', s)],
         '--data-dir', stage(DATA, LOCAL / 'stage' / 'data'), '--clip-dir', stage(A2_NEXAR, LOCAL / 'stage' / 'a2_nexar'),
         '--output-dir', local, *resume], log=LOCAL / 'train.log')
    last = metrics_last(local / 'metrics.jsonl')
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'J10 locally: {last["global_step"]}'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, RUN / name)
    flush_drive()
    assert ckpt_step(CKPT) == STEPS, f'Drive copy is not step {STEPS}'
    (LOCAL / 'verified.json').write_text(json.dumps({'step': STEPS, 'ckpt_bytes': CKPT.stat().st_size}))
    publish(LOCAL / 'verified.json', RUN / 'verified.json')
    print(f'trained in {(time.time() - t0) / 60:.0f} min | mil {last.get("mil")} | kim_mil {last.get("kim_mil")} '
          f'| W_out {last.get("kim_w_out_norm")} | proto cos {last.get("kim_proto_cos")}')
check_run()

## 6. K5 (whole) zero-shot: DoTA-CAP-dev (protocol B), DADA (whole), TAD, DAD — K5's protocols and bakes (§16 WH5)

In [ ]:
L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
L_TAD = LOCAL / 'TAD_meta'
L_TAD.mkdir(parents=True, exist_ok=True)
for f in TAD.iterdir():                     # json / txt only -- never a frame folder
    if f.is_file() and f.suffix in ('.json', '.txt'):
        shutil.copy2(f, L_TAD / f.name)

out = EVAL / 'dota' / K5
if not (out / 'clip_aucs.json').exists():
    local = LOCAL / 'eval' / 'dota'
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', CKPT,
         '--input-dir', stage(A2_DOTA, LOCAL / 'stage' / 'cap', CAP_IDS), '--s1-dir', stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS),
         '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
         '--data-dir', stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels'), '--split', CAP_DEV,
         '--out-dir', local, '--device', 'auto'], log=LOCAL / 'dota.log')
    publish(local, out)
print(f"DoTA-CAP-dev: macro {json.loads((out / 'protocol_b_readout.json').read_text())['macro']['mean']:.4f}")

out = EVAL / 'dada' / K5
if not (out / 'clip_aucs.json').exists():
    local = LOCAL / 'eval' / 'dada'
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', CKPT, '--t2-dir', L_T2,
         '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', stage(A2_DADA, LOCAL / 'stage' / 'dada', DADA_IDS),
         '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / 'dada.log')
    publish(local, out)
print(f"DADA whole: macro {json.loads((out / 'dada_readout.json').read_text())['macro']['mean']:.4f}")


def read_evaluate(name, dataset, data_dir, bake, ids, out):
    if not (out / 'results.json').exists():
        local = LOCAL / 'eval' / name
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.evaluate', '--ckpt', CKPT, '--set', f'data.dataset={dataset}',
             '--data-dir', data_dir, '--clip-dir', stage(bake, LOCAL / 'stage' / f'{name}_bake', ids),
             '--score-norm', 'auto', '--save-scores', '--output-dir', local], log=LOCAL / f'{name}.log')
        publish(local, out)
    r = json.loads((out / 'results.json').read_text())
    assert r['score_norm'] == 'none' and r['num_videos'] == len(ids), r
    print(f"{name}: macro {r['auc_macro']:.4f} over {r['auc_macro_videos']} | micro {r['auc']:.4f} | AP {r['ap']:.4f}")


read_evaluate('tad', TAD_NAME, L_TAD, A2_TAD, TAD_IDS, EVAL / 'tad' / K5 / f's{SEED}')
read_evaluate('dad', DAD_NAME, stage(DAD_DATA, LOCAL / 'stage' / 'dad_meta'), A2_DAD, DAD_IDS,
              EVAL / constants.DAD_OUTPUT_SET / K5 / f's{SEED}')
flush_drive()

## 7. `nexar_val` (in-domain for both rows; `--final` is never passed)

In [ ]:
from core.tools.nexar_eval import ITEM_AUCS_JSON, READOUT_JSON  # noqa: E402

L_X = stage(NEXAR_X, LOCAL / 'stage' / 'nexar_x', NEXAR_IDS)
L_U = stage(NEXAR_U, LOCAL / 'stage' / 'nexar_u', NEXAR_IDS)
L_STATS = stage(A2_NEXAR, LOCAL / 'stage' / 'nexar_stats', [])     # manifest + statistics only
ROWS = {WHOLE: (CKPT, EVAL / 'nexar' / WHOLE)}
if not ((K5_NEXAR / READOUT_JSON).exists() and (K5_NEXAR / ITEM_AUCS_JSON).exists()):
    K5_NEXAR = EVAL / 'nexar' / K5                                  # §13's read is gone: read K5 here
    ROWS[K5] = (K5_RUN / 'checkpoint_last.pt', K5_NEXAR)
for row, (ckpt, out) in ROWS.items():
    if not ((out / READOUT_JSON).exists() and (out / ITEM_AUCS_JSON).exists()):
        local = LOCAL / 'eval' / 'nexar' / row
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.nexar_eval', '--run', f's{SEED}', ckpt, '--stats-dir', L_STATS,
             '--video-s1-dir', L_U, '--census', CENSUS, '--clip-s1-dir', L_X,
             '--data-dir', L_DEFS, '--split', NEXAR_SPLIT, '--out-dir', local], log=LOCAL / 'nexar.log')
        publish(local, out)
for row, out in ((K5, K5_NEXAR), (WHOLE, EVAL / 'nexar' / WHOLE)):
    r = json.loads((out / READOUT_JSON).read_text())
    assert r['split'] == NEXAR_SPLIT and r['videos'] == len(NEXAR_IDS), (r['split'], r['videos'])
    print(f"{row}: crops macro {r['crops']['mean_over_placements']['macro']['mean']:.4f} | "
          f"whole macro {r['whole']['macro']['mean']:.4f} | clip AUC {r['whole']['clip_auc_max']:.4f}")
flush_drive()

## 8. The table -> **paste `REPORTS/k5_whole_report.md` to Claude**

In [ ]:
local = LOCAL / 'report'
pilot = ['--pilot-dir', PILOT] if (PILOT / READOUT_JSON).exists() else []
run([sys.executable, '-m', 'core.tools.k5_whole_report', '--kim-root', KIM, '--whole-root', ROOT,
     '--k5-nexar-dir', K5_NEXAR, '--seed', SEED, '--out-dir', local, *pilot])
publish(local, REPORTS)
print((REPORTS / 'k5_whole_report.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/k5_whole_report.md to Claude')